In [ ]:
import time, threading, multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

N_WORKERS = 4

def cpu_task(n):
    """CPU-bound: hitung banyak operasi. Terhalang GIL jika pakai thread."""
    return sum(i * i for i in range(n))

def io_task(_):
    """I/O-bound: simulasi menunggu jaringan/disk. GIL dilepas saat sleep."""
    time.sleep(1)

def timed(label, fn):
    t = time.perf_counter(); fn()
    print(f"  {label:<18}: {time.perf_counter() - t:5.2f} detik")

def run(executor_cls, task, args):
    with executor_cls(max_workers=N_WORKERS) as ex:
        list(ex.map(task, args))

def experiment(title, task, args):
    print(f"\n{title}")
    timed("Sequential", lambda: [task(a) for a in args])
    timed("Multithreading", lambda: run(ThreadPoolExecutor, task, args))
    timed("Multiprocessing", lambda: run(ProcessPoolExecutor, task, args))

# --- (C) Memori: thread berbagi, process terpisah ---
counter = 0
def add_ten():
    global counter
    counter += 10

def memory_demo():
    global counter
    counter = 0
    t = threading.Thread(target=add_ten); t.start(); t.join()
    print(f"  Setelah thread  : counter = {counter}  (berubah -> memori dibagi)")
    counter = 0
    p = mp.Process(target=add_ten); p.start(); p.join()
    print(f"  Setelah process : counter = {counter}  (tetap -> memori terpisah)")

if __name__ == "__main__":   # wajib: macOS memakai 'spawn', bukan 'fork'
    print(f"CPU core: {mp.cpu_count()} | worker: {N_WORKERS}")
    experiment("(A) CPU-bound: 4 tugas hitung berat", cpu_task, [5_000_000] * N_WORKERS)
    experiment("(B) I/O-bound: 4 tugas tunggu 1 detik", io_task, range(N_WORKERS))
    print("\n(C) Berbagi memori")
    memory_demo()

CPU core: 2 | worker: 4

(A) CPU-bound: 4 tugas hitung berat
  Sequential        :  3.52 detik
  Multithreading    :  3.46 detik
  Multiprocessing   :  2.14 detik

(B) I/O-bound: 4 tugas tunggu 1 detik
  Sequential        :  4.00 detik
  Multithreading    :  1.00 detik
  Multiprocessing   :  1.03 detik

(C) Berbagi memori
  Setelah thread  : counter = 10  (berubah -> memori dibagi)
  Setelah process : counter = 0  (tetap -> memori terpisah)
